In [ ]:
import pandas as pd
from pathlib import Path
from tabulate import tabulate

# Robust to whichever working directory Jupyter starts in (repo root vs.
# the notebook's own folder, which is VSCode's default for .ipynb files).
_CANDIDATES = [
    Path("datasets/ViNumQA/test.json"),
    Path("../../../datasets/ViNumQA/test.json"),
]
DATA_PATH = next((p for p in _CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not find test.json. cwd={Path.cwd()}, tried: {[str(p) for p in _CANDIDATES]}"
    )
print(f"Loading data from: {DATA_PATH.resolve()}")

df = pd.read_json(DATA_PATH)
df.sample(n=5)

In [3]:
len(df)

497

In [4]:
def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

def processing_input_question(sample):
    return sample["qa"]["question"]

def processing_program_content(sample):
    return sample["qa"]["program"]

def processing_answer_content(sample):
    return sample["qa"]["exe_ans"]

df["pre_text_processed"] = df.apply(lambda x: formatting_pre_text(x), axis=1)
df["post_text_processed"] = df.apply(lambda x: formatting_post_text(x), axis=1)
df["table_processed"] = df.apply(lambda x: formatting_table(x), axis=1)
df["input_question"] = df.apply(lambda x: processing_input_question(x), axis=1)
df["program_processed"] = df.apply(lambda x: processing_program_content(x), axis=1)
df["answer_processed"] = df.apply(lambda x: processing_answer_content(x), axis=1)
df.sample(n=5)

,pre_text,table,post_text,id,qa,pre_text_processed,post_text_processed,table_processed,input_question,program_processed,answer_processed
450,[kqkd 9m19 của pme đã ghi nhận chứng kiến về t...,"[[FY (Dec.), 12/15, 12/16, 12/17, 12/18, 12/19...",[.],masvn/2020/2020036-MAS_VN_PME_Initiate_Nov_14/...,{'question': 'Doanh thu của PME tăng bao nhiêu...,kqkd 9m19 của pme đã ghi nhận chứng kiến về tă...,.,| FY (Dec.) | 12/15 | 12/16 ...,Doanh thu của PME tăng bao nhiêu phần trăm từ ...,"subtract(1781, 1675), divide(#0, 1675)",0.06328
253,[mục lục tính thời vụ hoạt động kinh doanh của...,"[[, quý một, quý hai, quý ba, quý bốn], [2015,...",[doanh thu quý bốn năm 2015 bao gồm tác động c...,ALLE/2015/page_24.pdf-1,"{'question': 'Xét năm 2015, tỷ lệ phần trăm dự...",mục lục tính thời vụ hoạt động kinh doanh của ...,doanh thu quý bốn năm 2015 bao gồm tác động cả...,| | quý một | quý hai | quý ba ...,"Xét năm 2015, tỷ lệ phần trăm dự phòng cho các...","divide(2.8, 15.2)",0.18421
425,[chúng tôi đánh giá công ty cổ phần đầu tư và ...,"[[Tên dự án, Giá trị hiện tại thuần (triệu đồn...",[.],masvn/2020/2020044-CTCP_a__utu_va_Pha_ttrie__n...,{'question': 'Tỷ lệ nợ vay trên tổng giá trị d...,chúng tôi đánh giá công ty cổ phần đầu tư và p...,.,| Tên dự án | Giá trị hiện t...,Tỷ lệ nợ vay trên tổng giá trị doanh nghiệp là...,"divide(1070451, 2494125)",0.42919
280,[hoạt động chính trong lĩnh vực sản xuất thuốc...,"[[(Tỷ đồng), FY 2016, FY 2017, FY 2018, FY 201...",[.],masvn/2020/2020192-LTG_Companynote_MAS15.12.20...,{'question': 'Tính tổng doanh thu từ năm 2016 ...,hoạt động chính trong lĩnh vực sản xuất thuốc ...,.,| (Tỷ đồng) | FY 2016 | FY 2017 |...,Tính tổng doanh thu từ năm 2016 đến năm 2021(F...,"table_sum(Doanh thu, none)",47014.0
428,"[kết quả kinh doanh 1h19, kqkd 1h19 của dhg ch...","[[FY (Dec.), 2015, 2016, 2017, 2018, 2019F], [...",[.],masvn/2020/2020030-MAS_Pharmaceutical_Industry...,{'question': 'Tính trung bình biên lợi nhuận h...,kết quả kinh doanh 1h19\nkqkd 1h19 của dhg chứ...,.,| FY (Dec.) | 2015 | 2016 | 2017 ...,Tính trung bình biên lợi nhuận hoạt động (OP M...,"table_average(OP Margin (%), none)",0.1904


In [ ]:
df = df[["pre_text_processed", "table_processed", "post_text_processed", "input_question", "program_processed", "answer_processed"]]
# NOTE: use a flat list here, not [[...]] — the double brackets create a
# MultiIndex column (each name becomes a 1-tuple like ('question',)), which
# silently breaks boolean filtering like df["generated_program"] == "" later
# (df["col"] returns a 1-column DataFrame instead of a Series in that case).
df.columns = ["pre_text", "table", "post_text", "question", "program", "answer"]
df["generated_program"] = ""
df["calculated_program"] = ""
df

In [6]:
SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.
 
### LIST OF 10 VALID OPERATORS:
 
1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(val1, val2, val3, ...) -> sum of the values
8. table_average(val1, val2, val3, ...) -> arithmetic mean of the values
9. table_max(val1, val2, val3, ...) -> maximum of the values
10. table_min(val1, val2, val3, ...) -> minimum of the values
 
### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- Do not use square brackets `[]` inside table-type functions (e.g. write table_max(1, 2, 3), not table_max([1, 2, 3])).
- Do not perform mental calculations or provide explanations. The output must contain only the program string.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use 'none'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}
 
[TABLE]
{table}
 
[TEXT AFTER TABLE]
{post_text}
 
### QUESTION:
{question}
 
### PROGRAM:"""

In [ ]:
import base64
import requests
import io
import os
from io import BytesIO
from PIL import Image
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.environ["API_KEY"]
BASE_URL = os.environ["BASE_URL"]
MODEL = "DeepSeek-V4-Flash" # Llama-3.3-70B-Instruct; DeepSeek-V4-Flash; GLM-5.1

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

In [ ]:
# Reasoning-model detection.
# Reasoning models (DeepSeek-R1 style, "...-Thinking", QwQ, o1/o3, etc.) emit an
# internal chain-of-thought before the final program string, so they need a much
# larger token budget than a plain instruct model or the response gets truncated
# (content=None / empty string, as happened in the earlier run of this notebook).
REASONING_MODEL_KEYWORDS = ["r1", "thinking", "reasoner", "qwq", "o1", "o3", "reasoning"]

def is_reasoning_model(model_name: str) -> bool:
    name = model_name.lower()
    return any(kw in name for kw in REASONING_MODEL_KEYWORDS)

IS_REASONING = is_reasoning_model(MODEL)

# Manual overrides: the name-based heuristic above is unreliable for models
# whose naming doesn't hint at reasoning (verify with the probe cell below,
# which checks for a `reasoning_content`/`reasoning` field on the real API
# response, then add the model name here once confirmed).
# - DeepSeek-V4-Flash: confirmed reasoning via `reasoning_content` field.
# NOTE: GLM-5.1 was tested and excluded -- its inference on this endpoint
# gets stuck in a degenerate token-repetition loop (never reaches `content`,
# even at max_tokens=32768 with temperature/frequency_penalty tuning). This
# looks like a server-side deployment bug, not something fixable client-side.
REASONING_MODEL_OVERRIDES = {"DeepSeek-V4-Flash": True}
if MODEL in REASONING_MODEL_OVERRIDES:
    IS_REASONING = REASONING_MODEL_OVERRIDES[MODEL]

MAX_TOKENS = 8192 if IS_REASONING else 512

print(f"Model: {MODEL} | reasoning_model={IS_REASONING} | max_tokens={MAX_TOKENS}")

In [ ]:
# Empirical check (more reliable than name-guessing): make one real call and
# inspect the raw response. Reasoning-model APIs (DeepSeek-R1 style) return the
# chain-of-thought in a separate field on the message (name varies by vendor:
# "reasoning_content" for DeepSeek, "reasoning" for GLM), distinct from the
# final `content`. If that field is present and non-empty, it IS a reasoning
# model regardless of what IS_REASONING guessed from the name.
_probe = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": USER_MESSAGE_FRAME.format(
            pre_text="Doanh thu năm 2023 là 914, năm 2022 là 391.",
            table="", post_text="",
            question="Doanh thu năm 2023 gấp bao nhiêu lần năm 2022?"
        )},
    ],
    temperature=0.0,
    max_tokens=MAX_TOKENS,
)
_msg = _probe.choices[0].message
_reasoning_field = getattr(_msg, "reasoning_content", None) or getattr(_msg, "reasoning", None)

print("finish_reason:", _probe.choices[0].finish_reason)
print("content:", repr(_msg.content))
print("reasoning field present:", _reasoning_field is not None and bool(str(_reasoning_field).strip()))
if _reasoning_field:
    print("reasoning (first 300 chars):", str(_reasoning_field)[:300])

In [ ]:
import time
from pathlib import Path
from tqdm import tqdm

# Checkpointed, retrying generation loop.
# - Retries on transient errors / empty responses instead of crashing the whole run.
# - Saves progress to CSV every CHECKPOINT_EVERY samples and resumes from it,
#   so a crash (like the AttributeError at sample 483 in the first run) doesn't
#   throw away already-generated predictions.
CHECKPOINT_PATH = Path(f"outputs/0shot_{MODEL}_checkpoint.csv")
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
CHECKPOINT_EVERY = 20

if CHECKPOINT_PATH.exists():
    ckpt = pd.read_csv(CHECKPOINT_PATH).fillna("")
    df.loc[ckpt.index, "generated_program"] = ckpt["generated_program"].values
    print(f"Resumed {(df['generated_program'] != '').sum()} / {len(df)} samples from checkpoint.")

def call_model(pre_text, table, post_text, question, max_retries=3):
    user_msg = USER_MESSAGE_FRAME.format(
        pre_text=pre_text, table=table, post_text=post_text, question=question
    )
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_MESSAGE},
                    {"role": "user", "content": user_msg},
                ],
                temperature=0.0,
                max_tokens=MAX_TOKENS,
                stream=False,
            )
            choice = resp.choices[0]
            content = choice.message.content
            # A reasoning model can burn its whole token budget on the hidden
            # chain-of-thought and get cut off (finish_reason == "length")
            # before finishing (or even starting) the final program string.
            if content is not None and content.strip() and choice.finish_reason != "length":
                return content.strip().strip("\n")
        except Exception as e:
            if attempt == max_retries:
                print(f"[row failed after {max_retries} attempts] {e}")
                return ""
        if attempt < max_retries:
            time.sleep(2 * attempt)
    return ""

todo_idx = df.index[df["generated_program"] == ""]
for i, df_index in enumerate(tqdm(todo_idx, desc="Generating program...")):
    values = df.loc[df_index]
    output = call_model(values["pre_text"], values["table"], values["post_text"], values["question"])
    df.at[df_index, "generated_program"] = output

    if (i + 1) % CHECKPOINT_EVERY == 0:
        df.to_csv(CHECKPOINT_PATH, index=False)

df.to_csv(CHECKPOINT_PATH, index=False)
print(f"Done. {(df['generated_program'] != '').sum()} / {len(df)} samples generated.")

In [ ]:
"""
Parser + evaluation utilities for FinQA/VLSP-2025 style computation programs.

Program string format (as seen in the dataset / notebook), e.g.:
    "add(30, 1)"
    "subtract(9829, 642), divide(#0, 642)"
    "table_sum(-54, 30, 1), table_sum(-23, 292)"

- Steps are separated by top-level commas (commas NOT inside parentheses).
- Each step is `operator(arg1, arg2, ...)`.
- An arg can be a literal number, or a reference to a previous step's
  result using `#0`, `#1`, ... (0-indexed).
- `none` is treated as a missing/undefined value.
- A literal ending in `%` (e.g. "16.63%") represents a percentage and is
  converted to its decimal fraction (16.63% -> 0.1663), matching how the
  dataset's gold programs and exe_ans values are computed.

Two metrics are provided, matching the paper's definitions:
- Program Accuracy (PA): structural match between the generated program
  and the gold program (operator sequence + normalized numeric args).
- Execution Accuracy (EA): whether executing the generated program
  produces the same final numeric value as the gold answer.

NOTE on PA: FinQA-style datasets often have multiple valid gold programs
for the same question (see `program_re` in the paper). This implementation
only compares against a single gold program string. If you have alternative
gold programs available, run `compute_program_accuracy` against each and
take the max.
"""

import re
from typing import List, Tuple, Union, Optional


VALID_OPERATORS = {
    "add", "subtract", "multiply", "divide", "exp", "greater",
    "table_sum", "table_average", "table_max", "table_min",
}

_BINARY_OPS = {"add", "subtract", "multiply", "divide", "exp", "greater"}
_TABLE_OPS = {"table_sum", "table_average", "table_max", "table_min"}

_NUM_RE = re.compile(r"^-?\d+(\.\d+)?$")
_REF_RE = re.compile(r"^#(\d+)$")


def _parse_numeric_literal(raw: str) -> float:
    """
    Clean and parse a numeric literal, correctly handling a trailing '%' by
    dividing by 100 (a plain string-strip of '%' silently produces a value
    100x too large, e.g. treating "16.63%" as 16.63 instead of 0.1663 --
    which breaks execution/comparison against exe_ans, which is always a
    plain decimal fraction).
    Raises ValueError if the cleaned string isn't a valid number.
    """
    cleaned = raw.replace(",", "").replace("$", "").strip()
    is_percent = cleaned.endswith("%")
    if is_percent:
        cleaned = cleaned[:-1].strip()
    if not _NUM_RE.match(cleaned):
        raise ValueError(f"Cannot parse numeric argument: '{raw}'")
    value = float(cleaned)
    return value / 100.0 if is_percent else value


# --------------------------------------------------------------------------
# 1. Extraction: pull a program string out of raw (possibly messy) model output
# --------------------------------------------------------------------------

def extract_program(raw_text: str) -> str:
    """
    Extract a clean program string from raw LLM output that may contain
    markdown fences, extra prose, or trailing whitespace/newlines.

    Strategy: find all substrings matching `operator(...)` for the 10 known
    operators (no nested parens expected within a single step) and join them
    with ", " in the order they appear. Falls back to the stripped raw text
    if no operator call is found.
    """
    text = raw_text.strip()
    text = re.sub(r"```[a-zA-Z]*", "", text)
    text = text.replace("```", "").strip()

    op_names = "|".join(sorted(VALID_OPERATORS, key=len, reverse=True))
    pattern = rf"\b(?:{op_names})\([^()]*\)"
    matches = re.findall(pattern, text)

    if matches:
        return ", ".join(m.strip() for m in matches)
    return text


# --------------------------------------------------------------------------
# 2. Parsing: program string -> list of (operator, [args])
# --------------------------------------------------------------------------

def _split_top_level(s: str, sep: str = ",") -> List[str]:
    """Split a string on `sep`, but only at depth 0 (i.e. outside parens)."""
    parts = []
    depth = 0
    current = []
    for ch in s:
        if ch == "(":
            depth += 1
            current.append(ch)
        elif ch == ")":
            depth -= 1
            current.append(ch)
        elif ch == sep and depth == 0:
            parts.append("".join(current))
            current = []
        else:
            current.append(ch)
    if current:
        parts.append("".join(current))
    return [p.strip() for p in parts if p.strip() != ""]


def parse_program(program_str: str) -> List[Tuple[str, List[str]]]:
    """
    Parse a program string into a list of (operator, raw_args) steps.
    Raises ValueError if the string is malformed or uses an unknown operator.
    """
    program_str = program_str.strip().rstrip(",").strip()
    if not program_str:
        raise ValueError("Empty program string.")

    steps_raw = _split_top_level(program_str, sep=",")

    # Steps are comma-separated at top level, but each step itself is
    # "op(args...)" where args are also comma separated -> the naive split
    # above would break "op(a, b)" into "op(a" and " b)". So instead we
    # scan for complete `name(...)` chunks directly.
    steps: List[Tuple[str, List[str]]] = []
    call_pattern = re.compile(r"\s*([a-zA-Z_]+)\(([^()]*)\)\s*")
    pos = 0
    text = program_str
    while pos < len(text):
        m = call_pattern.match(text, pos)
        if not m:
            raise ValueError(f"Malformed program near: '{text[pos:pos+30]}...'")
        op = m.group(1).strip()
        if op not in VALID_OPERATORS:
            raise ValueError(f"Unknown operator: '{op}'")
        args = _split_top_level(m.group(2), sep=",")
        steps.append((op, args))
        pos = m.end()
        if pos < len(text) and text[pos] == ",":
            pos += 1

    if not steps:
        raise ValueError("No valid steps parsed.")
    return steps


# --------------------------------------------------------------------------
# 3. Execution: resolve args (numbers or #N references) and run each step
# --------------------------------------------------------------------------

def _resolve_arg(arg: str, results: List[float]) -> Optional[float]:
    arg = arg.strip()

    ref_match = _REF_RE.match(arg)
    if ref_match:
        idx = int(ref_match.group(1))
        if idx >= len(results):
            raise ValueError(f"Reference #{idx} used before step {idx} was computed.")
        return results[idx]

    if arg.lower() == "none":
        return None

    return _parse_numeric_literal(arg)


def execute_program(program_str: str) -> Tuple[float, List[float]]:
    """
    Parse and execute a program string.
    Returns (final_result, list_of_all_intermediate_results).
    Raises ValueError on malformed programs, unknown operators, missing
    (`none`) operands where a real value is required, or arithmetic errors
    (e.g. division by zero).
    """
    steps = parse_program(program_str)
    results: List[float] = []

    for op, raw_args in steps:
        vals = [_resolve_arg(a, results) for a in raw_args]

        if op in _BINARY_OPS:
            if len(vals) != 2:
                raise ValueError(f"Operator '{op}' expects 2 args, got {len(vals)}.")
            a, b = vals
            if a is None or b is None:
                raise ValueError(f"Operator '{op}' received a 'none' operand.")
            if op == "add":
                r = a + b
            elif op == "subtract":
                r = a - b
            elif op == "multiply":
                r = a * b
            elif op == "divide":
                if b == 0:
                    raise ValueError("Division by zero.")
                r = a / b
            elif op == "exp":
                r = a ** b
            elif op == "greater":
                r = 1.0 if a > b else 0.0

        elif op in _TABLE_OPS:
            if len(vals) < 1:
                raise ValueError(f"Operator '{op}' requires at least 1 arg.")
            if any(v is None for v in vals):
                raise ValueError(f"Operator '{op}' received a 'none' operand.")
            if op == "table_sum":
                r = sum(vals)
            elif op == "table_average":
                r = sum(vals) / len(vals)
            elif op == "table_max":
                r = max(vals)
            elif op == "table_min":
                r = min(vals)
        else:
            raise ValueError(f"Unknown operator: '{op}'")

        results.append(r)

    return results[-1], results


# --------------------------------------------------------------------------
# 4. Metrics
# --------------------------------------------------------------------------

def _normalize_program(program_str: str) -> List[Tuple[str, Tuple[str, ...]]]:
    """
    Normalize a parsed program for structural comparison:
    - operator names kept as-is
    - numeric args parsed to their true value (percent literals divided by
      100, see _parse_numeric_literal) and rounded to 6 decimals, so "30"
      == "30.0" but "16.63%" != "16.63" (they're not the same magnitude)
    - #N references kept as-is (order-sensitive)
    """
    steps = parse_program(program_str)
    normalized = []
    for op, args in steps:
        norm_args = []
        for a in args:
            a = a.strip()
            if _REF_RE.match(a) or a.lower() == "none":
                norm_args.append(a.lower())
            else:
                try:
                    norm_args.append(f"{round(_parse_numeric_literal(a), 6)}")
                except ValueError:
                    norm_args.append(a)
        normalized.append((op, tuple(norm_args)))
    return normalized


def compute_program_accuracy(
    generated_program: str,
    gold_program: str,
    alternative_gold_programs: Optional[List[str]] = None,
) -> float:
    """
    Program Accuracy (PA): 1.0 if the generated program structurally matches
    the gold program (same operator sequence, same args in the same order,
    numeric args compared after normalization), else 0.0.

    If `alternative_gold_programs` is provided (e.g. FinQA's `program_re`
    field), the generated program is also compared against each alternative
    and the best (max) match is returned.

    Returns 0.0 if either program fails to parse.
    """
    try:
        gen_norm = _normalize_program(generated_program)
    except ValueError:
        return 0.0

    gold_candidates = [gold_program] + (alternative_gold_programs or [])
    for gold in gold_candidates:
        try:
            gold_norm = _normalize_program(gold)
        except ValueError:
            continue
        if gen_norm == gold_norm:
            return 1.0
    return 0.0


def compute_execution_accuracy(
    generated_program: str,
    gold_answer: Union[str, float],
    rel_tol: float = 1e-3,
    abs_tol: float = 1e-4,
) -> float:
    """
    Execution Accuracy (EA): 1.0 if executing the generated program yields
    a numeric value equal to `gold_answer` within tolerance, else 0.0.

    `gold_answer` may be a string (as in the dataset's `exe_ans` field,
    e.g. "31.0") or a float. Returns 0.0 if the program fails to parse
    or execute, or if the gold answer can't be parsed as a number.
    """
    try:
        predicted, _ = execute_program(generated_program)
    except (ValueError, ZeroDivisionError, OverflowError):
        return 0.0

    try:
        gold = _parse_numeric_literal(str(gold_answer))
    except ValueError:
        return 0.0

    if abs(predicted - gold) <= max(abs_tol, rel_tol * abs(gold)):
        return 1.0
    return 0.0


# --------------------------------------------------------------------------
# 5. Batch evaluation helper (for a pandas DataFrame)
# --------------------------------------------------------------------------

def evaluate_dataframe(
    df,
    generated_col: str = "generated_program",
    gold_program_col: str = "program_processed",
    gold_answer_col: str = "answer_processed",
    alternative_gold_col: Optional[str] = None,
    extract_first: bool = True,
):
    """
    Compute per-row PA/EA and overall averages for a DataFrame.

    Adds two columns to a copy of `df`: 'pa_score' and 'ea_score', and
    returns (df_with_scores, {"program_accuracy": ..., "execution_accuracy": ...}).

    Set `extract_first=True` (default) to run `extract_program` on the raw
    generated text first, useful when the model output may contain extra
    prose, markdown fences, etc.
    """
    df = df.copy()
    pa_scores = []
    ea_scores = []

    for _, row in df.iterrows():
        raw_generated = row[generated_col]
        generated = extract_program(raw_generated) if extract_first else raw_generated
        gold_program = row[gold_program_col]
        gold_answer = row[gold_answer_col]

        alt_programs = None
        if alternative_gold_col is not None and alternative_gold_col in row:
            alt_val = row[alternative_gold_col]
            if isinstance(alt_val, str) and alt_val.strip():
                alt_programs = [alt_val]

        pa = compute_program_accuracy(generated, gold_program, alt_programs)
        ea = compute_execution_accuracy(generated, gold_answer)

        pa_scores.append(pa)
        ea_scores.append(ea)

    df["pa_score"] = pa_scores
    df["ea_score"] = ea_scores

    summary = {
        "program_accuracy": sum(pa_scores) / len(pa_scores) if pa_scores else 0.0,
        "execution_accuracy": sum(ea_scores) / len(ea_scores) if ea_scores else 0.0,
    }
    return df, summary


# --------------------------------------------------------------------------
# Quick self-test using the example from the dataset
# --------------------------------------------------------------------------

if __name__ == "__main__":
    # Example from the uploaded sample: program "add(30, 1)" -> exe_ans "31.0"
    gold_program = "add(30, 1)"
    gold_answer = "31.0"

    # Case 1: exact match
    gen1 = "add(30, 1)"
    print("PA (exact):", compute_program_accuracy(gen1, gold_program))
    print("EA (exact):", compute_execution_accuracy(gen1, gold_answer))

    # Case 2: numerically correct but structurally different (from the notebook run)
    gen2 = "table_sum(-54, 30, 1), table_sum(-23, 292)"
    print("PA (wrong structure):", compute_program_accuracy(gen2, gold_program))
    print("EA (wrong structure):", compute_execution_accuracy(gen2, gold_answer))

    # Case 3: messy raw output with extra text/fences
    raw = "```\nSure, here is the program:\nadd(30, 1)\n```"
    extracted = extract_program(raw)
    print("Extracted:", extracted)
    print("PA (extracted):", compute_program_accuracy(extracted, gold_program))

    # Case 4: percent literals must be divided by 100, not just stripped of '%'
    gold_pct_program = "subtract(16.63%, 14.23%)"
    gold_pct_answer = "0.024"
    gen_pct = "subtract(16.63%, 14.23%)"
    print("EA (percent, exact):", compute_execution_accuracy(gen_pct, gold_pct_answer))
    print("PA (percent, exact):", compute_program_accuracy(gen_pct, gold_pct_program))
    gen_pct_wrong_magnitude = "subtract(16.63, 14.23)"  # model dropped the '%'
    print("PA (percent vs dropped '%', should NOT match):",
          compute_program_accuracy(gen_pct_wrong_magnitude, gold_pct_program))

In [ ]:
df_scored, summary = evaluate_dataframe(
    df,
    generated_col="generated_program",   # cột bạn đang ghi output model vào
    gold_program_col="program",
    gold_answer_col="answer",
)

print(summary)  # {'program_accuracy': ..., 'execution_accuracy': ...}

In [ ]:
import json

results_path = Path(f"outputs/0shot_{MODEL}_results.csv")
summary_path = Path(f"outputs/0shot_{MODEL}_summary.json")

df_scored.to_csv(results_path, index=False)
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump({"model": MODEL, "shot": "0-shot", "max_tokens": MAX_TOKENS, **summary}, f, ensure_ascii=False, indent=2)

print(f"Saved per-sample results to {results_path}")
print(f"Saved summary to {summary_path}")